# Summarizing text using LangChain

Let's summarize the book _Moby Dick_. Load the text into a variable:

In [1]:
with open("./data/Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

## First chain: Split the document

In [4]:
from langchain_ollama import ChatOllama
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

In [5]:
llm = ChatOllama(model="granite4.1:3b")

In [6]:
# Break the document into chunks of the specified size
text_chunks_chain = (
    RunnableLambda(lambda x:
        [
            {
                'chunk': text_chunk,
            }
            for text_chunk in TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

## Second chain: Setting up the map chain

In [7]:
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

In [8]:
summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)

In [9]:
summarize_chunk_chain = summarize_chunk_prompt | llm

In [10]:
summarize_map_chain = (
    RunnableParallel(
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

## Third chain: Setting up the reduce chain

In [11]:
summarize_summaries_prompt_template = """
Write a concise summary of the following text,
which joins several summaries, and include the main details.
Text {summaries}
"""

In [12]:
summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)

In [13]:
summarize_reduce_chain = (
    RunnableLambda(lambda x:
    {
        'summaries': '\n'.join([i['summary'] for i in x])                   
    })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

## The document-splitting chain: MapReduce combined chain

In [14]:
map_reduce_chain = (
    text_chunks_chain # Split chain
    | summarize_map_chain.map() # Map chain
    | summarize_reduce_chain # Reduce chain
)

## MapReduce execution

In [15]:
summary = map_reduce_chain.invoke(moby_dick_book)
print(summary)

"Moby-Dick; or The Whale" is a novel by Herman Melville, originally published as Project Gutenberg eBook #2701 in 2001. It follows Ishmael's journey on a whaling voyage from Nantucket to New Bedford, driven by his desire for adventure and connection with nature through the ocean. The narrative explores themes of isolation, identity, and humanity's relationship with the natural world, particularly focusing on the significance of water.

The text details Ishmael's stay at "The Spouter Inn" in Nantucket, a dimly lit establishment where he observes young seamen examining exotic items from around the world. He encounters Bulkington, a tall, heavily built man with a Southern accent, who raises suspicions about his character and potential danger due to cultural differences.

A nighttime encounter reveals Queequeg, a harpooneer with a tattooed face and unusual appearance, performing ritualistic actions that initially frighten Ishmael. The landlord reassures him that Queequeg poses no threat, a

## Summarizing across documents
_Refine technique_, this technique a final summary is constructed incrementally by iteratively summarizing the combination of the current final summary and one of the document chunks.

Let's summarize content from four different sources: a Wikipedia page and a set of files in various formats (TXT,DOCX,PDF) stored in a local folder.

In [3]:
# Wikipedia content
import wikipedia
from langchain_community.document_loaders import WikipediaLoader

# Forzamos un User-Agent para que Wikipedia no rechace la conexión
wikipedia.set_user_agent("Building-LLM-Applications/1.0 (santiago@tu-email.com)")

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [4]:
wikipedia_docs

[Document(metadata={'title': 'Paestum', 'summary': 'Paestum ( PEST-əm, US also  PEE-stəm, Latin: [ˈpae̯stũː]) was a major ancient Greek city on the coast of the Tyrrhenian Sea, in Magna Graecia. The ruins of Paestum are famous for their three ancient Greek temples in the Doric order dating from about 550 to 450 BCE that are in an excellent state of preservation. The city walls and amphitheatre are largely intact, and the bottom of the walls of many other structures remain, as well as paved roads. The site is open to the public, and there is a modern national museum within it, which also contains the finds from the associated Greek site of Foce del Sele.\nPaestum was established around 600 BCE by settlers from Sybaris, a Greek colony in southern Italy, under the name of Poseidonia (Ancient Greek: Ποσειδωνία). The city thrived as a Greek settlement for about two centuries, witnessing the development of democracy. In 400 BCE, the Lucanians seized the city. Romans took over in 273 BCE, ren

In [9]:
# File-based content
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("data/Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("data/Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("data/Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [10]:
# Creating the Document list
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

With everything compiled, you're ready to summarize the content using the refine technique. The next step is to create a chain that generates the final summary.

In [22]:
# Progressively refining the final summary
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="gemma2:2b")

In [23]:
# Define the chain with the related prompt for summarizing individual documents
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)
doc_summary_chain = doc_summary_prompt | llm

In [24]:
# Set up the chain for refining the summary by iteratively combining 
# the current summary with the summary of an additional document:
refine_summary_template = """
You must produce a final summary from the current refined summary
which has been generated so far and from the content of an
additional document.
This is the current refined summary generated so far:
{current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)
refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [25]:
# Define a function that loops over each document, summarizes it using the doc_summary_chain
# and refines the overall summary using the refine_chain
def refine_summary(docs):
    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = {
            "current_refined_summary": current_refined_summary,
            "text": doc.page_content
        }
        intermediate_steps.append(intermediate_step)

        current_refined_summary = refine_chain.invoke(intermediate_step)

    return {"final_summary": current_refined_summary, "intermediate_steps": intermediate_steps}

In [27]:
# Start the summarization process by calling refine_summary() on your prepared document list
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': '## Summary of Paestum\n\nPaestum was an ancient Greek city located on the Tyrrhenian coast in Magna Graecia, renowned for its three Doric-order temples dating back to the 5th and 4th century BCE. The city\'s history includes: \n\n* **Founding & Growth (600 BCE - 400 BCE):**  Paestum was founded around 600 BCE and flourished before being captured by Lucanians in 400 BCE. The Romans established a Latin colony there in 273 BCE renaming it Paestum.\n* **Decline & Abandonment (400 BCE - Present):** Factors like trade shifts, flooding, and other factors led to the site\'s abandonment after declining around the 400 BCE.  \n* **Rediscovery (18th Century - Present):** Paestum was rediscovered in the 18th century and is now open to the public. Archaeological research continues today.\n\nThe city\'s location in Campania, Italy, features surrounding hamlets and mentions historical figures. It shares geographical proximity with Capaccio, another area in the same region. The name 